In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="multimodal",
    notebook_name="01_audio_language.ipynb"
)

# Audio-Language Models: From Sound Waves to Text

Every voice assistant, transcription service, and subtitle generator starts with the same trick: turn sound into a picture, then process that picture with a neural network.

In this notebook, we will build the audio processing pipeline from scratch:

1. **Generate a synthetic audio signal** — create sound waves from pure tones
2. **Compute a spectrogram** — turn sound into a picture using the Fourier transform
3. **Build a mel filter bank** — match human hearing perception
4. **Create a mel-spectrogram** — the actual input to Whisper and other speech models
5. **Visualize the full pipeline** — waveform → spectrogram → mel-spectrogram

**Prerequisites:** NumPy basics, what a neural network does at a high level ([00-neural-networks](../../00-neural-networks/)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")

## Part 1: What Is Sound?

Sound is vibrations in air. A microphone converts these vibrations into a list of numbers — each number is the air pressure at one moment in time.

The **sample rate** is how many numbers we record per second. CD quality is 44,100 Hz. Whisper uses 16,000 Hz — enough for speech but less data to process.

A pure tone is a single frequency — like a tuning fork. Let's create one.

In [ ]:
# Create a pure tone (sine wave)
sample_rate = 16000  # 16kHz, same as Whisper
duration = 0.5       # 0.5 seconds
frequency = 440      # A4 note (440 Hz)

# Time axis
t = np.arange(0, duration, 1/sample_rate)
print(f"Duration: {duration}s at {sample_rate} Hz = {len(t)} samples")

# Generate sine wave: amplitude * sin(2π * frequency * time)
signal = np.sin(2 * np.pi * frequency * t)

print(f"Signal shape: {signal.shape}")
print(f"First 10 values: {np.round(signal[:10], 4)}")
print(f"Min: {signal.min():.2f}, Max: {signal.max():.2f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full signal
axes[0].plot(t * 1000, signal, color='#2196F3', linewidth=0.5)
axes[0].set_xlabel('Time (ms)', fontsize=12)
axes[0].set_ylabel('Amplitude', fontsize=12)
axes[0].set_title(f'Pure Tone: {frequency} Hz (A4 note) — full 0.5 seconds', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Zoomed in to see individual waves
zoom_samples = int(0.01 * sample_rate)  # 10ms
axes[1].plot(t[:zoom_samples] * 1000, signal[:zoom_samples], color='#2196F3', linewidth=2)
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_ylabel('Amplitude', fontsize=12)
axes[1].set_title(f'Zoomed: first 10ms — each wave = 1/{frequency}s = {1000/frequency:.1f}ms', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOne cycle of {frequency} Hz takes {1000/frequency:.2f} ms.")
print(f"In 10ms, we see {frequency * 0.01:.1f} complete cycles.")

### Real sounds are combinations of frequencies

A pure tone has one frequency. Real sounds — voices, music, noise — are combinations of many frequencies at once. Let's create a more realistic signal.

In [ ]:
# Create a signal with multiple frequencies (like a chord)
# This simulates a more realistic sound

f1, f2, f3 = 261.63, 329.63, 392.00  # C4, E4, G4 (C major chord)

signal_c = 0.5 * np.sin(2 * np.pi * f1 * t)  # C4
signal_e = 0.3 * np.sin(2 * np.pi * f2 * t)  # E4 (softer)
signal_g = 0.2 * np.sin(2 * np.pi * f3 * t)  # G4 (even softer)

chord = signal_c + signal_e + signal_g

# Also create a signal that changes over time (two notes in sequence)
half = len(t) // 2
sequential = np.zeros_like(t)
sequential[:half] = np.sin(2 * np.pi * 440 * t[:half])      # A4 for first half
sequential[half:] = np.sin(2 * np.pi * 880 * t[half:])      # A5 for second half

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Individual components
zoom = int(0.02 * sample_rate)  # 20ms
for ax, sig, name, color in [
    (axes[0, 0], signal_c[:zoom], f'C4 ({f1:.0f} Hz)', '#F44336'),
    (axes[0, 0], signal_e[:zoom], f'E4 ({f2:.0f} Hz)', '#4CAF50'),
    (axes[0, 0], signal_g[:zoom], f'G4 ({f3:.0f} Hz)', '#2196F3'),
]:
    ax.plot(t[:zoom] * 1000, sig, color=color, linewidth=1.5, label=name)
axes[0, 0].legend(fontsize=10)
axes[0, 0].set_title('Individual Frequencies', fontsize=13)
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].grid(True, alpha=0.3)

# Combined chord
axes[0, 1].plot(t[:zoom] * 1000, chord[:zoom], color='#9C27B0', linewidth=1.5)
axes[0, 1].set_title('Combined: C Major Chord', fontsize=13)
axes[0, 1].set_xlabel('Time (ms)')
axes[0, 1].grid(True, alpha=0.3)

# Sequential signal
axes[1, 0].plot(t * 1000, sequential, color='#FF9800', linewidth=0.5)
axes[1, 0].set_title('Sequential: A4 (440 Hz) then A5 (880 Hz)', fontsize=13)
axes[1, 0].set_xlabel('Time (ms)')
axes[1, 0].axvline(x=duration/2 * 1000, color='red', linestyle='--', alpha=0.5, label='Note change')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Challenge: looking at the chord waveform, can you tell what frequencies are in it?
axes[1, 1].plot(t[:zoom] * 1000, chord[:zoom], color='#9C27B0', linewidth=1.5)
axes[1, 1].set_title('Can you see 3 frequencies in this wave?\n(No — that is why we need spectrograms!)', fontsize=13)
axes[1, 1].set_xlabel('Time (ms)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The combined waveform hides its component frequencies.")
print("We need the Fourier transform to reveal them.")

## Part 2: The Fourier Transform — Revealing Hidden Frequencies

The Fourier transform answers the question: **which frequencies are present in this signal, and how loud is each one?**

It takes a time-domain signal (amplitude over time) and converts it to a frequency-domain representation (amplitude at each frequency).

We use NumPy's FFT (Fast Fourier Transform) — the same algorithm used in every audio processing system.

In [ ]:
# Apply FFT to the chord signal
fft_result = np.fft.rfft(chord)  # rfft = real FFT (only positive frequencies)
frequencies = np.fft.rfftfreq(len(chord), 1/sample_rate)

# Magnitude spectrum (how loud each frequency is)
magnitude = np.abs(fft_result) / len(chord)  # Normalize by signal length

print(f"FFT output shape: {fft_result.shape}")
print(f"Frequency range: {frequencies[0]:.0f} Hz to {frequencies[-1]:.0f} Hz")
print(f"Frequency resolution: {frequencies[1] - frequencies[0]:.1f} Hz per bin")

# Find peaks
peak_indices = np.argsort(magnitude)[-5:][::-1]
print(f"\nTop 5 frequency peaks:")
for idx in peak_indices:
    print(f"  {frequencies[idx]:>8.1f} Hz  (magnitude: {magnitude[idx]:.4f})")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(frequencies, magnitude, color='#2196F3', linewidth=1.5)

# Mark the known frequencies
for freq, name, color in [(f1, 'C4', '#F44336'), (f2, 'E4', '#4CAF50'), (f3, 'G4', '#FF9800')]:
    idx = np.argmin(np.abs(frequencies - freq))
    ax.scatter(frequencies[idx], magnitude[idx], color=color, s=100, zorder=5)
    ax.annotate(f'{name}\n({freq:.0f} Hz)', (frequencies[idx], magnitude[idx]),
                textcoords='offset points', xytext=(10, 10), fontsize=11, color=color)

ax.set_xlabel('Frequency (Hz)', fontsize=13)
ax.set_ylabel('Magnitude', fontsize=13)
ax.set_title('FFT reveals the three frequencies hidden in the chord!', fontsize=14)
ax.set_xlim(0, 1000)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nThe FFT perfectly recovers the three frequencies we mixed together.")
print("This is the foundation of all audio analysis.")

## Part 3: The Spectrogram — Frequency Over Time

The FFT tells us which frequencies are in the **entire** signal. But speech changes over time — the letter 'S' has different frequencies than the letter 'O'.

We need to know which frequencies are present **at each moment in time**. This is the **Short-Time Fourier Transform (STFT)**:

1. Slide a short window across the audio
2. Apply FFT to each window
3. Stack the results → a 2D picture (time × frequency)

This picture is a **spectrogram**.

In [ ]:
def compute_stft(signal, window_size=400, hop_size=160, sample_rate=16000):
    """Compute the Short-Time Fourier Transform.
    
    Args:
        signal: 1D audio signal
        window_size: number of samples per window (25ms at 16kHz = 400)
        hop_size: number of samples between windows (10ms at 16kHz = 160)
        sample_rate: samples per second
    
    Returns:
        spectrogram: 2D array (n_freq_bins, n_time_frames)
        times: time axis in seconds
        freqs: frequency axis in Hz
    """
    # Hann window reduces spectral leakage at window edges
    window = np.hanning(window_size)
    
    # Number of time frames
    n_frames = (len(signal) - window_size) // hop_size + 1
    
    # Number of frequency bins (positive frequencies only)
    n_freq = window_size // 2 + 1
    
    # Compute STFT frame by frame
    spectrogram = np.zeros((n_freq, n_frames))
    
    for i in range(n_frames):
        start = i * hop_size
        frame = signal[start:start + window_size] * window
        fft_frame = np.fft.rfft(frame)
        spectrogram[:, i] = np.abs(fft_frame) ** 2  # Power spectrum
    
    times = np.arange(n_frames) * hop_size / sample_rate
    freqs = np.fft.rfftfreq(window_size, 1/sample_rate)
    
    return spectrogram, times, freqs


# Compute spectrogram of the sequential signal (A4 then A5)
spec, times, freqs = compute_stft(sequential, window_size=400, hop_size=160)

print(f"STFT parameters:")
print(f"  Window size: 400 samples = {400/sample_rate*1000:.0f}ms")
print(f"  Hop size: 160 samples = {160/sample_rate*1000:.0f}ms")
print(f"  Frequency bins: {spec.shape[0]}")
print(f"  Time frames: {spec.shape[1]}")
print(f"  Spectrogram shape: {spec.shape}")
print(f"  Frequency range: {freqs[0]:.0f} Hz to {freqs[-1]:.0f} Hz")
print(f"  Frequency resolution: {freqs[1]-freqs[0]:.1f} Hz per bin")

In [ ]:
# Visualize: waveform + spectrogram side by side
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Waveform
axes[0].plot(t * 1000, sequential, color='#FF9800', linewidth=0.5)
axes[0].set_ylabel('Amplitude', fontsize=12)
axes[0].set_title('Waveform: A4 (440 Hz) then A5 (880 Hz)', fontsize=14)
axes[0].axvline(x=duration/2 * 1000, color='red', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

# Spectrogram (log scale for visibility)
spec_db = 10 * np.log10(spec + 1e-10)  # Convert to decibels
im = axes[1].pcolormesh(times * 1000, freqs, spec_db, cmap='magma', shading='auto')
axes[1].set_ylabel('Frequency (Hz)', fontsize=12)
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_title('Spectrogram: bright spots show which frequencies are present at each moment', fontsize=14)
axes[1].set_ylim(0, 2000)  # Zoom to relevant range
axes[1].axvline(x=duration/2 * 1000, color='white', linestyle='--', alpha=0.5)

# Annotate the two notes
axes[1].annotate('440 Hz (A4)', xy=(100, 440), fontsize=12, color='white',
                 fontweight='bold')
axes[1].annotate('880 Hz (A5)', xy=(300, 880), fontsize=12, color='white',
                 fontweight='bold')

plt.colorbar(im, ax=axes[1], label='Power (dB)')
plt.tight_layout()
plt.show()

print("The spectrogram clearly shows:")
print("  - 440 Hz is present in the first half (bright spot at 440 Hz, 0-250ms)")
print("  - 880 Hz is present in the second half (bright spot at 880 Hz, 250-500ms)")
print("  - The waveform alone does not show this clearly — the spectrogram does!")

### A More Complex Example

Let's create a signal that simulates speech-like frequency changes — multiple frequencies changing over time.

In [ ]:
# Create a more complex signal: a frequency sweep + harmonics
duration_long = 1.0  # 1 second
t_long = np.arange(0, duration_long, 1/sample_rate)

# Frequency sweep from 200 Hz to 800 Hz (like a rising pitch)
freq_sweep = 200 + 600 * (t_long / duration_long)  # Linear sweep
sweep = np.sin(2 * np.pi * np.cumsum(freq_sweep) / sample_rate)

# Add a steady harmonic at 1000 Hz
harmonic = 0.3 * np.sin(2 * np.pi * 1000 * t_long)

# Add some noise
noise = 0.05 * np.random.randn(len(t_long))

complex_signal = sweep + harmonic + noise

# Compute spectrogram
spec_complex, times_c, freqs_c = compute_stft(complex_signal)
spec_complex_db = 10 * np.log10(spec_complex + 1e-10)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t_long * 1000, complex_signal, color='#2196F3', linewidth=0.3)
axes[0].set_ylabel('Amplitude', fontsize=12)
axes[0].set_title('Waveform: Frequency sweep (200→800 Hz) + steady 1000 Hz + noise', fontsize=14)
axes[0].grid(True, alpha=0.3)

im = axes[1].pcolormesh(times_c * 1000, freqs_c, spec_complex_db, cmap='magma', shading='auto')
axes[1].set_ylabel('Frequency (Hz)', fontsize=12)
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_title('Spectrogram reveals the structure hidden in the waveform', fontsize=14)
axes[1].set_ylim(0, 2000)
plt.colorbar(im, ax=axes[1], label='Power (dB)')

plt.tight_layout()
plt.show()

print("The spectrogram clearly shows:")
print("  - A rising diagonal line: the frequency sweep from 200 to 800 Hz")
print("  - A horizontal line at 1000 Hz: the steady harmonic")
print("  - Faint background: the noise")
print("\nThis is exactly how speech looks in a spectrogram — formants (resonant")
print("frequencies of the vocal tract) create patterns that identify each sound.")

## Part 4: The Mel Scale — Hearing Like a Human

The spectrogram treats all frequencies equally: 100 Hz and 200 Hz get the same spacing as 7,000 Hz and 7,100 Hz.

But human hearing does not work this way. You can easily tell the difference between a low bass note and a slightly higher bass note. But two high-pitched sounds that are 100 Hz apart sound almost the same.

The **mel scale** fixes this. It spaces frequencies according to how humans hear them — more detail at low frequencies, less at high frequencies.

In [ ]:
def hz_to_mel(freq):
    """Convert frequency in Hz to mel scale."""
    return 2595 * np.log10(1 + freq / 700)

def mel_to_hz(mel):
    """Convert mel scale back to Hz."""
    return 700 * (10 ** (mel / 2595) - 1)


# Show the mel scale
hz_range = np.linspace(0, 8000, 1000)
mel_range = hz_to_mel(hz_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hz to Mel curve
axes[0].plot(hz_range, mel_range, color='#4CAF50', linewidth=2.5)
axes[0].set_xlabel('Frequency (Hz)', fontsize=13)
axes[0].set_ylabel('Mel Scale', fontsize=13)
axes[0].set_title('Mel Scale: logarithmic above ~1kHz\n(compresses high frequencies)', fontsize=14)

# Mark some key points
for hz in [100, 500, 1000, 2000, 4000, 8000]:
    mel = hz_to_mel(hz)
    axes[0].plot([hz, hz, 0], [0, mel, mel], 'k--', alpha=0.3, linewidth=0.8)
    axes[0].annotate(f'{hz} Hz\n= {mel:.0f} mel', (hz, mel),
                     textcoords='offset points', xytext=(10, 5), fontsize=9)
axes[0].grid(True, alpha=0.3)

# Show perceptual spacing
hz_pairs = [(100, 200), (1000, 2000), (4000, 8000)]
mel_diffs = []
for h1, h2 in hz_pairs:
    m1, m2 = hz_to_mel(h1), hz_to_mel(h2)
    mel_diffs.append(m2 - m1)

x_pos = range(len(hz_pairs))
bars = axes[1].bar(x_pos, mel_diffs, color=['#F44336', '#FF9800', '#4CAF50'], width=0.6)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'{h1}→{h2} Hz' for h1, h2 in hz_pairs], fontsize=11)
axes[1].set_ylabel('Mel Difference', fontsize=13)
axes[1].set_title('Perceptual distance for equal Hz differences\n(100→200 Hz feels bigger than 4000→8000 Hz)', fontsize=14)

for bar, val in zip(bars, mel_diffs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:.0f} mel', ha='center', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Doubling from 100→200 Hz = 134 mel (big perceptual change)")
print("Doubling from 4000→8000 Hz = 731 mel (still noticeable but less dramatic)")
print("\nThe mel scale gives more resolution where humans hear better (low frequencies).")

### Building a Mel Filter Bank

To convert a regular spectrogram into a mel-spectrogram, we create a set of **triangular filters** spaced evenly on the mel scale. Each filter covers a range of frequencies and outputs one number: how much energy is in that range.

In [ ]:
def create_mel_filterbank(n_mels=80, n_fft=400, sample_rate=16000):
    """Create a mel-scale triangular filter bank.
    
    Args:
        n_mels: number of mel bands (Whisper uses 80)
        n_fft: FFT size
        sample_rate: audio sample rate
    
    Returns:
        filterbank: (n_mels, n_fft//2 + 1) matrix
    """
    # Frequency range
    f_min = 0
    f_max = sample_rate / 2  # Nyquist frequency
    
    # Convert min/max to mel scale
    mel_min = hz_to_mel(f_min)
    mel_max = hz_to_mel(f_max)
    
    # Create evenly spaced points on mel scale
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    
    # Convert back to Hz
    hz_points = mel_to_hz(mel_points)
    
    # Convert Hz to FFT bin indices
    n_freq = n_fft // 2 + 1
    bin_points = np.round(hz_points * n_fft / sample_rate).astype(int)
    bin_points = np.clip(bin_points, 0, n_freq - 1)
    
    # Create triangular filters
    filterbank = np.zeros((n_mels, n_freq))
    
    for m in range(n_mels):
        left = bin_points[m]
        center = bin_points[m + 1]
        right = bin_points[m + 2]
        
        # Rising slope: left to center
        if center > left:
            filterbank[m, left:center] = np.linspace(0, 1, center - left, endpoint=False)
        
        # Falling slope: center to right
        if right > center:
            filterbank[m, center:right] = np.linspace(1, 0, right - center, endpoint=False)
        
        # Peak
        if center < n_freq:
            filterbank[m, center] = 1.0
    
    return filterbank


# Create and visualize the filter bank
n_mels = 80
filterbank = create_mel_filterbank(n_mels=n_mels, n_fft=400, sample_rate=sample_rate)

print(f"Filter bank shape: {filterbank.shape}")
print(f"  {filterbank.shape[0]} mel bands x {filterbank.shape[1]} frequency bins")

# Plot a subset of filters
fig, ax = plt.subplots(figsize=(14, 5))
freq_axis = np.fft.rfftfreq(400, 1/sample_rate)

# Show every 10th filter
colors = plt.cm.viridis(np.linspace(0, 0.9, 8))
for i, color in zip(range(0, n_mels, 10), colors):
    ax.plot(freq_axis, filterbank[i], color=color, linewidth=1.5, label=f'Mel band {i}')

ax.set_xlabel('Frequency (Hz)', fontsize=13)
ax.set_ylabel('Filter Weight', fontsize=13)
ax.set_title(f'Mel Filter Bank ({n_mels} filters)\nTriangles are narrow at low freq (more detail) and wide at high freq (less detail)',
             fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Low-frequency filters are narrow: they capture fine pitch differences.")
print("High-frequency filters are wide: they blur frequencies that sound similar to us.")
print("This matches how the human ear works.")

## Part 5: The Mel-Spectrogram — Whisper's Input

Now we put it all together. The mel-spectrogram is computed by multiplying the filter bank with the power spectrogram, then taking the log.

This is exactly what Whisper does before feeding audio to its encoder.

In [ ]:
def compute_mel_spectrogram(signal, sample_rate=16000, n_mels=80,
                            window_size=400, hop_size=160):
    """Compute the log mel-spectrogram (Whisper's input format).
    
    Pipeline: audio → STFT → power spectrum → mel filter → log
    
    Returns:
        log_mel: (n_mels, n_frames) log mel-spectrogram
        times: time axis
    """
    # Step 1: STFT
    power_spec, times, freqs = compute_stft(signal, window_size, hop_size, sample_rate)
    
    # Step 2: Create mel filter bank
    fb = create_mel_filterbank(n_mels, window_size, sample_rate)
    
    # Step 3: Apply mel filters
    # (n_mels, n_freq) @ (n_freq, n_frames) = (n_mels, n_frames)
    mel_spec = fb @ power_spec
    
    # Step 4: Log scale (compress dynamic range)
    log_mel = np.log(mel_spec + 1e-10)
    
    return log_mel, times


# Compute mel-spectrogram for the complex signal
mel_spec, mel_times = compute_mel_spectrogram(complex_signal)

print(f"Mel-spectrogram shape: {mel_spec.shape}")
print(f"  {mel_spec.shape[0]} mel bands x {mel_spec.shape[1]} time frames")
print(f"  This is what Whisper's encoder receives!")

In [ ]:
# The full pipeline visualization: waveform → spectrogram → mel-spectrogram

# Recompute regular spectrogram for comparison
spec_regular, times_r, freqs_r = compute_stft(complex_signal)
spec_regular_db = 10 * np.log10(spec_regular + 1e-10)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Waveform
axes[0].plot(t_long * 1000, complex_signal, color='#2196F3', linewidth=0.3)
axes[0].set_ylabel('Amplitude', fontsize=12)
axes[0].set_title('Step 1: Raw Waveform (what the microphone records)', fontsize=14)
axes[0].grid(True, alpha=0.3)

# 2. Regular spectrogram
im1 = axes[1].pcolormesh(times_r * 1000, freqs_r, spec_regular_db,
                          cmap='magma', shading='auto')
axes[1].set_ylabel('Frequency (Hz)', fontsize=12)
axes[1].set_title('Step 2: Spectrogram (STFT — linear frequency scale)', fontsize=14)
axes[1].set_ylim(0, 3000)
plt.colorbar(im1, ax=axes[1], label='Power (dB)', shrink=0.8)

# 3. Mel-spectrogram
im2 = axes[2].pcolormesh(mel_times * 1000, np.arange(mel_spec.shape[0]),
                          mel_spec, cmap='magma', shading='auto')
axes[2].set_ylabel('Mel Band', fontsize=12)
axes[2].set_xlabel('Time (ms)', fontsize=12)
axes[2].set_title('Step 3: Mel-Spectrogram (mel scale — matches human hearing)', fontsize=14)
plt.colorbar(im2, ax=axes[2], label='Log Power', shrink=0.8)

plt.tight_layout()
plt.show()

print("Compare the spectrogram (linear scale) and mel-spectrogram (mel scale):")
print("  - Low frequencies get more detail in the mel version")
print("  - High frequencies are compressed")
print("  - The mel-spectrogram is smaller (80 bands vs 201 bins)")
print(f"\nData compression: {spec_regular.shape[0]}x{spec_regular.shape[1]}")
print(f"  → {mel_spec.shape[0]}x{mel_spec.shape[1]} ({mel_spec.size / spec_regular.size:.1%} of the data)")

## Part 6: From Mel-Spectrogram to Text (Encoder-Decoder Pattern)

In a real speech recognition system like Whisper, the mel-spectrogram is fed into a transformer encoder-decoder. Let's build a simplified version to see the pattern.

We will not build a full Whisper model (that would require GPU training on 680,000 hours of audio). Instead, we will show the encoder-decoder pattern with a simple sequence-to-sequence task: converting a sequence of mel frames into a sequence of output tokens.

In [ ]:
# Simplified encoder-decoder to show the architecture pattern
# This is NOT a working speech recognizer — it shows how the pieces connect

def simplified_encoder(mel_input, W_enc):
    """Simulates the encoder: projects mel features to hidden states.
    
    Real Whisper: 2 conv layers + N transformer layers
    Our version: single linear projection
    """
    # mel_input shape: (n_mels, n_frames) -> transpose to (n_frames, n_mels)
    x = mel_input.T  # (n_frames, n_mels)
    # Project to hidden dimension
    hidden = x @ W_enc  # (n_frames, d_hidden)
    return hidden


def simplified_cross_attention(decoder_state, encoder_output):
    """Simulates cross-attention: decoder looks at encoder output.
    
    Real Whisper: multi-head cross-attention
    Our version: dot-product attention
    """
    # decoder_state: (1, d_hidden) — current decoder position
    # encoder_output: (n_frames, d_hidden) — all encoder frames
    
    scores = decoder_state @ encoder_output.T  # (1, n_frames)
    # Softmax to get attention weights
    scores = scores - scores.max()  # Numerical stability
    weights = np.exp(scores) / np.exp(scores).sum()
    # Weighted sum of encoder output
    context = weights @ encoder_output  # (1, d_hidden)
    return context, weights


# Set up dimensions
n_mels_demo = 80
d_hidden = 16
n_frames = mel_spec.shape[1]

np.random.seed(42)
W_enc = np.random.randn(n_mels_demo, d_hidden) * 0.1

# Run encoder
encoder_output = simplified_encoder(mel_spec, W_enc)

print(f"Encoder:")
print(f"  Input (mel-spectrogram): ({mel_spec.shape[0]}, {mel_spec.shape[1]}) = (mels, frames)")
print(f"  Output (hidden states): {encoder_output.shape} = (frames, d_hidden)")

# Simulate decoder step: generate attention over encoder
# Pretend we are generating the 3rd output token
decoder_state = np.random.randn(1, d_hidden) * 0.1
context, attn_weights = simplified_cross_attention(decoder_state, encoder_output)

print(f"\nDecoder (one step):")
print(f"  Decoder state: {decoder_state.shape}")
print(f"  Cross-attention weights: {attn_weights.shape} (one weight per encoder frame)")
print(f"  Context vector: {context.shape}")
print(f"  The decoder 'looks at' specific parts of the audio via cross-attention")

# Visualize cross-attention
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(mel_times * 1000, attn_weights.flatten(), color='#4CAF50', linewidth=1.5)
ax.fill_between(mel_times * 1000, 0, attn_weights.flatten(), alpha=0.3, color='#4CAF50')
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Attention Weight', fontsize=12)
ax.set_title('Cross-Attention: where the decoder "looks" in the audio\n(this is how Whisper aligns text to audio)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nIn real Whisper:")
print("  - The encoder processes the full mel-spectrogram once")
print("  - The decoder generates tokens one at a time")
print("  - At each step, cross-attention lets the decoder focus on the relevant audio")
print("  - This is how the model aligns audio to text without explicit alignment labels")

## Summary

In this notebook, we built the audio processing pipeline from scratch:

| Step | What We Did | Key Insight |
|------|------------|-------------|
| 1. Sound waves | Generated synthetic audio signals | Sound is just a list of numbers (air pressure samples) |
| 2. Fourier Transform | Revealed hidden frequencies with FFT | Any signal is a sum of pure tones at different frequencies |
| 3. STFT + Spectrogram | Split audio into windows, FFT each one | Shows which frequencies are present at each moment in time |
| 4. Mel scale | Converted to perceptual frequency spacing | Matches how humans hear — more detail at low frequencies |
| 5. Mel filter bank | Built triangular filters on the mel scale | Compresses 201 frequency bins to 80 mel bands |
| 6. Mel-spectrogram | Applied filters + log scaling | This is Whisper's actual input format |
| 7. Encoder-decoder | Showed the architecture pattern | Encoder processes audio, decoder generates text via cross-attention |

### The Full Pipeline

```
Raw audio (480,000 samples)
    → STFT (201 freq bins × 3000 frames)
    → Mel filter bank (80 mel bands × 3000 frames)
    → Log scaling
    → Encoder (transformer, outputs hidden states)
    → Decoder (generates text tokens via cross-attention)
```

**Next:** For experiments on window size, mel scale, and noise robustness, see [01_audio_language_experiments.ipynb](./01_audio_language_experiments.ipynb). For interview-depth math and Q&A, see [audio-language-interview.md](./audio-language-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)